In [ ]:
# Importar librerías estándar para análisis de datos y visualización
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Herramientas de Scikit-learn: división de datos, escalado y métricas de clasificación
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# TensorFlow / Keras para construir y entrenar la red neuronal
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping

### Análisis Exploratorio

In [ ]:
# Ruta al dataset de clasificación de cáncer
file_path = '~/tensorflow_ejemplos/DATA/cancer_classification.csv'

In [ ]:
# Cargar el CSV en un DataFrame de pandas y mostrar información general
df = pd.read_csv(file_path)
df.info()

In [ ]:
# Estadísticas descriptivas de todas las variables numéricas
df.describe().transpose()

In [ ]:
# Visualizar el balance entre clases (benigno vs maligno)
sns.countplot(data=df, x='benign_0__mal_1');

In [ ]:
# Mapa de calor de correlaciones entre todas las variables
sns.heatmap(df.corr());

In [ ]:
# Correlación de cada feature con la variable objetivo, ordenada de menor a mayor
df.corr()['benign_0__mal_1'].sort_values()

In [ ]:
# Gráfico de barras de correlaciones (excluyendo la variable consigo misma)
df.corr()['benign_0__mal_1'][:-1].sort_values().plot(kind='bar');

### Escalado y Train-Test Split

In [ ]:
# Separar features (X) y variable objetivo (y)
X = df.drop('benign_0__mal_1', axis=1).values
y = df['benign_0__mal_1'].values

In [ ]:
# Dividir en entrenamiento (80%) y prueba (20%) con semilla para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

In [ ]:
# Escalar features al rango [0, 1] usando solo el conjunto de entrenamiento
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)   # Ajustar y transformar train
X_test = scaler.transform(X_test)         # Transformar test con los parámetros de train

In [ ]:
# Forma del conjunto de entrenamiento: (muestras, features)
X_train.shape

In [ ]:
# Forma del conjunto de prueba
X_test.shape

### Crear Modelos de Entrenamiento

#### Ejemplo 1: Overfitting

In [ ]:
# Definir arquitectura de la red neuronal usando la API Sequential moderna de Keras
# Se obtiene el número de features dinámicamente para no hardcodear dimensiones
input_dim = X_train.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),               # Capa de entrada: shape = (features,)
    Dense(units=32, activation='relu'),      # Capa oculta 1: 32 neuronas + ReLU
    Dense(units=16, activation='relu'),      # Capa oculta 2: 16 neuronas + ReLU
    Dense(units=1, activation='sigmoid')     # Capa de salida: probabilidad con sigmoid (clasificación binaria)
])

In [ ]:
# Compilar el modelo: binary_crossentropy es la loss estándar para clasificación binaria
# metrics=['accuracy'] permite monitorear la precisión durante el entrenamiento
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Entrenar el modelo; usamos validation_data para monitorear overfitting en cada época
history = model.fit(
    x=X_train,
    y=y_train,
    validation_data=(X_test, y_test),
    batch_size=128,
    epochs=500
)

In [ ]:
# Convertir el historial de entrenamiento a DataFrame y graficar la evolución de la pérdida
history_df = pd.DataFrame(history.history)
history_df.plot();

#### Ejemplo 2: Early Stop y Mejores Pesos

In [ ]:
# EarlyStopping: detiene el entrenamiento si la pérdida en validación no mejora en 50 épocas
# restore_best_weights=True recupera los pesos de la mejor época
early_stop = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

In [ ]:
model = Sequential([
    Input(shape=(input_dim,)),               # Capa de entrada: shape = (features,)
    Dense(units=32, activation='relu'),      # Capa oculta 1: 32 neuronas + ReLU
    Dense(units=16, activation='relu'),      # Capa oculta 2: 16 neuronas + ReLU
    Dense(units=1, activation='sigmoid')     # Capa de salida: probabilidad con sigmoid (clasificación binaria)
])

In [ ]:
# Compilar el modelo con métrica de accuracy para monitorear desempeño
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Entrenar el modelo con callback de EarlyStopping para evitar sobreajuste
history = model.fit(
    x=X_train,
    y=y_train,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    batch_size=128,
    epochs=500
)

In [ ]:
# Graficar la evolución de loss y accuracy durante el entrenamiento
history_df = pd.DataFrame(history.history)
history_df.plot();

#### Ejemplo 3: Dropout

In [ ]:
# EarlyStopping con paciencia de 50 épocas y recuperación de mejores pesos
early_stop = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)

In [ ]:
model = Sequential([
    Input(shape=(input_dim,)),               # Capa de entrada: shape = (features,)
    Dense(units=32, activation='relu'),      # Capa oculta 1: 32 neuronas + ReLU
    Dropout(0.3),                            # Apagar aleatoriamente 30% de neuronas durante entrenamiento
    Dense(units=16, activation='relu'),      # Capa oculta 2: 16 neuronas + ReLU
    Dropout(0.3),                            # Regularización adicional con Dropout
    Dense(units=1, activation='sigmoid')     # Capa de salida: probabilidad con sigmoid (clasificación binaria)
])

In [ ]:
# Compilar el modelo con métrica de accuracy
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Entrenar el modelo con EarlyStopping y Dropout para reducir sobreajuste
history = model.fit(
    x=X_train,
    y=y_train,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    batch_size=128,
    epochs=500
)

In [ ]:
# Graficar la evolución de loss y accuracy durante el entrenamiento
history_df = pd.DataFrame(history.history)
history_df.plot();

### Evaluación

In [ ]:
# Generar predicciones y convertir probabilidades a clases usando umbral de 0.5
predictions = (model.predict(X_test) > 0.5).astype('int32')

In [ ]:
# Reporte de clasificación: precision, recall, f1-score por clase
print(classification_report(y_test, predictions))

In [ ]:
# Matriz de confusión: comparar predicciones vs valores reales
sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción')
plt.ylabel('Actual');

### Predicción

In [ ]:
# Seleccionar una muestra aleatoria del DataFrame original para validar la predicción
np.random.seed(42)
random_num = np.random.randint(len(df))
single_data = df.drop('benign_0__mal_1', axis=1).iloc[random_num]

In [ ]:
# Escalar la muestra individual con el mismo scaler usado en entrenamiento
single_data = scaler.transform(single_data.values.reshape(-1, input_dim))

In [ ]:
# Predecir la clase de la muestra seleccionada (0 = benigno, 1 = maligno)
prediction = (model.predict(single_data) > 0.5).astype('int32')
prediction

In [ ]:
# Mostrar los datos reales de la muestra para comparar con la predicción
df.iloc[random_num]